# Cache-Aware Scheduling

## Learning Objectives
1. Detect shared prefixes across requests
2. Implement KV cache reuse
3. Analyze cache hit rates and latency improvements
4. Handle copy-on-write for diverging sequences

In [ ]:
import numpy as np
import hashlib
import matplotlib.pyplot as plt
from collections import defaultdict
import time

np.random.seed(42)
print('Cache-aware request scheduling simulation')

## Level 1: Basic Prefix Hashing

In [ ]:
# Level 1: Detect prefix sharing with hashing
# Each request has a prefix (system prompt + context)

def hash_prefix(tokens, prefix_len=None):
    '''Compute hash of first k tokens.'''
    if prefix_len is None:
        prefix_len = len(tokens)
    else:
        prefix_len = min(prefix_len, len(tokens))
    
    prefix = tokens[:prefix_len]
    # Simple hash for demo (would use SHA-256 in practice)
    return hashlib.sha256(str(prefix).encode()).hexdigest()[:16]

# Simulate requests with shared prefixes
system_prompt = list(range(100))  # 100-token system prompt
doc_context_1 = list(range(100, 350))  # 250-token context
doc_context_2 = list(range(100, 200))  # 100-token context

# 10 requests, some share prefixes
requests = []
requests.append(system_prompt + doc_context_1 + list(range(1000, 1020)))  # Req 1
requests.append(system_prompt + doc_context_1 + list(range(1020, 1040)))  # Req 2 - shares prefix with 1
requests.append(system_prompt + doc_context_2 + list(range(1040, 1060)))  # Req 3 - shares system, diff context
requests.append(system_prompt + list(range(400, 450)) + list(range(1060, 1080)))  # Req 4 - different context

print('Generated 4 requests')
print(f'Req 1: {len(requests[0])} tokens, system+doc1+query')
print(f'Req 2: {len(requests[1])} tokens, system+doc1+query (shares system+doc1 with Req1)')
print(f'Req 3: {len(requests[2])} tokens, system+doc2+query (shares system with others)')
print(f'Req 4: {len(requests[3])} tokens, system+diff_context+query')

# Compute prefix hashes
prefix_len = 350  # System + one doc context
prefix_hashes = {}
for i, req in enumerate(requests):
    h = hash_prefix(req, prefix_len)
    prefix_hashes[i] = h
    print(f'\nRequest {i}: prefix_hash={h}, total_tokens={len(req)}')

# Group by prefix
groups = defaultdict(list)
for req_id, h in prefix_hashes.items():
    groups[h].append(req_id)

print(f'\nPrefix groups:')
for h, reqs in groups.items():
    print(f'  Hash {h}: Requests {reqs}')
    
print(f'\nCache reuse opportunity:')
for h, reqs in groups.items():
    if len(reqs) > 1:
        shared_tokens = 350
        saved_tokens = shared_tokens * (len(reqs) - 1)
        print(f'  Reqs {reqs}: {saved_tokens} tokens can reuse KV cache')

## Level 2: KV Cache Manager with Copy-on-Write

In [ ]:
# Level 2: KV Cache management with copy-on-write
# Simulate KV cache pages (like vLLM's PagedAttention)

class KVCacheBlock:
    '''Represents a block of KV cache (e.g., 16 tokens worth).'''
    def __init__(self, block_id, token_range, ref_count=1):
        self.block_id = block_id
        self.token_range = token_range  # (start, end) token indices
        self.ref_count = ref_count
        self.physical_memory = np.random.randn(16, 64, 2)  # 16 tokens, 64D, K+V
    
    def duplicate(self):
        '''Copy-on-write: create new physical block.'''
        new_block = KVCacheBlock(
            block_id=self.block_id,
            token_range=self.token_range,
            ref_count=1
        )
        self.ref_count -= 1
        return new_block
    
    def release(self):
        self.ref_count -= 1
        return self.ref_count == 0

class KVCacheManager:
    '''Manage KV cache blocks with reference counting.'''
    def __init__(self, max_blocks=100):
        self.max_blocks = max_blocks
        self.blocks = {}
        self.block_counter = 0
        self.free_blocks = []
    
    def allocate_block(self, token_range):
        '''Allocate new KV cache block.'''
        if self.free_blocks:
            block_id = self.free_blocks.pop()
        else:
            block_id = self.block_counter
            self.block_counter += 1
        
        block = KVCacheBlock(block_id, token_range)
        self.blocks[block_id] = block
        return block_id
    
    def reference(self, block_id):
        '''Increment reference count (reuse from another request).'''
        if block_id in self.blocks:
            self.blocks[block_id].ref_count += 1
    
    def release_block(self, block_id):
        '''Release a block.'''
        if block_id in self.blocks:
            if self.blocks[block_id].release():
                del self.blocks[block_id]
                self.free_blocks.append(block_id)
    
    def memory_usage(self):
        return len(self.blocks) * 16 * 64 * 2 * 4 / 1024 / 1024  # MB

# Simulate requests with cache reuse
cache = KVCacheManager()

# Request 1: allocate blocks for full sequence
print('Processing request 1:')
req1_prefix_blocks = [cache.allocate_block((0, 16)), cache.allocate_block((16, 32))]
req1_decode_blocks = [cache.allocate_block((32, 48))]
print(f'  Allocated {len(req1_prefix_blocks)} prefix blocks + {len(req1_decode_blocks)} decode blocks')
print(f'  Memory: {cache.memory_usage():.2f} MB')

# Request 2: reuse prefix blocks from request 1
print('\nProcessing request 2 (shares prefix with Req1):')
req2_prefix_blocks = req1_prefix_blocks.copy()  # Reference same blocks
for block_id in req2_prefix_blocks:
    cache.reference(block_id)
req2_decode_blocks = [cache.allocate_block((32, 48))]
print(f'  Reused {len(req2_prefix_blocks)} prefix blocks + allocated {len(req2_decode_blocks)} decode blocks')
print(f'  Memory: {cache.memory_usage():.2f} MB')

# Request 3: partial prefix reuse (system prompt only)
print('\nProcessing request 3 (shares system prompt with others):')
req3_prefix_blocks = [req1_prefix_blocks[0]]  # Reuse system prompt block
cache.reference(req3_prefix_blocks[0])
req3_new_blocks = [cache.allocate_block((16, 32))]  # New document block
req3_decode_blocks = [cache.allocate_block((32, 48))]
print(f'  Reused {len(req3_prefix_blocks)} block + allocated {len(req3_new_blocks) + len(req3_decode_blocks)} new blocks')
print(f'  Memory: {cache.memory_usage():.2f} MB')

# Memory savings
print(f'\nMemory efficiency:')
print(f'  Without cache reuse: ~{3 * (len(req1_prefix_blocks) + len(req1_decode_blocks)):.1f} MB')
print(f'  With cache reuse: {cache.memory_usage():.2f} MB')
print(f'  Savings: {(1 - cache.memory_usage() / (3*64)) * 100:.1f}%''')

## Real-World Example 1: RAG Workload

In [ ]:
# Example 1: RAG system with high prefix overlap
# Typical RAG: system_prompt + retrieved_doc + user_query

np.random.seed(42)

# Workload characteristics
n_requests = 100
system_prompt_len = 100  # Constant across all requests
doc_len = 300  # Retrieved documents
query_len = 50

# Many requests share the same top-k documents
doc_ids = np.random.randint(0, 20, n_requests)  # Only 20 unique docs
unique_doc_ids = np.unique(doc_ids)

# Simulate prefix hashing
prefix_cache = {}
cache_hits = 0
cache_misses = 0

latencies_no_cache = []
latencies_with_cache = []

for req_id in range(n_requests):
    doc_id = doc_ids[req_id]
    
    # Prefix = system_prompt + doc
    prefix = (0, doc_id)  # Tuple for hashability
    
    # Check cache
    if prefix in prefix_cache:
        cache_hits += 1
        kv_cache_latency = 0  # Already have it
        base_latency = query_len * 5  # Only decode query
    else:
        cache_misses += 1
        prefix_cache[prefix] = True
        kv_cache_latency = (system_prompt_len + doc_len) * 0.2  # Prefill latency
        base_latency = (system_prompt_len + doc_len) * 0.2 + query_len * 5
    
    latencies_no_cache.append(base_latency + np.random.normal(0, 10))
    latencies_with_cache.append(kv_cache_latency + query_len * 5 + np.random.normal(0, 5))

latencies_no_cache = np.array(latencies_no_cache)
latencies_with_cache = np.array(latencies_with_cache)

cache_hit_rate = cache_hits / n_requests

print(f'RAG Workload Analysis')
print('-' * 60)
print(f'Total requests: {n_requests}')
print(f'Unique documents: {len(unique_doc_ids)} / 20')
print(f'Cache hit rate: {cache_hit_rate*100:.1f}%')
print(f'\nLatency (ms):')
print(f'  Without cache - Mean: {latencies_no_cache.mean():.1f}, P99: {np.percentile(latencies_no_cache, 99):.1f}')
print(f'  With cache - Mean: {latencies_with_cache.mean():.1f}, P99: {np.percentile(latencies_with_cache, 99):.1f}')
print(f'\nLatency reduction:')
print(f'  Mean: {(1 - latencies_with_cache.mean() / latencies_no_cache.mean()) * 100:.1f}%')
print(f'  P99: {(1 - np.percentile(latencies_with_cache, 99) / np.percentile(latencies_no_cache, 99)) * 100:.1f}%''')

## Real-World Example 2: Multi-Turn Conversation

In [ ]:
# Example 2: Multi-turn conversations (each turn reuses history)
# Turn 1: user1 + bot1 (new)
# Turn 2: user1 + bot1 + user2 + bot2 (reuses user1+bot1)
# Turn 3: user1 + bot1 + user2 + bot2 + user3 + bot3 (reuses turns 1-2)

turns_per_conversation = 5
num_conversations = 20

total_latency_no_cache = 0
total_latency_with_cache = 0
total_tokens_no_cache = 0
total_tokens_with_cache = 0

conversation_turn_tokens = []

for conv_id in range(num_conversations):
    conversation_latency_no = 0
    conversation_latency_with = 0
    
    history_tokens = 0  # Accumulated conversation history
    
    for turn in range(1, turns_per_conversation + 1):
        user_tokens = np.random.poisson(20)
        bot_tokens = np.random.poisson(50)
        
        input_tokens = history_tokens + user_tokens
        output_tokens = bot_tokens
        
        # Latency without cache: process everything
        latency_no_cache = input_tokens * 0.2 + output_tokens * 5 + np.random.normal(0, 10)
        
        # Latency with cache: only process new user input, reuse history
        latency_with_cache = user_tokens * 0.2 + output_tokens * 5 + np.random.normal(0, 5)
        
        conversation_latency_no += latency_no_cache
        conversation_latency_with += latency_with_cache
        
        history_tokens += user_tokens + bot_tokens
        conversation_turn_tokens.append((turn, user_tokens + bot_tokens))
    
    total_latency_no_cache += conversation_latency_no
    total_latency_with_cache += conversation_latency_with
    total_tokens_no_cache += history_tokens * turns_per_conversation
    total_tokens_with_cache += sum(tokens for _, tokens in conversation_turn_tokens)

avg_latency_no = total_latency_no_cache / (num_conversations * turns_per_conversation)
avg_latency_with = total_latency_with_cache / (num_conversations * turns_per_conversation)

print(f'Multi-Turn Conversation Analysis')
print('-' * 60)
print(f'Conversations: {num_conversations}')
print(f'Turns per conversation: {turns_per_conversation}')
print(f'\nPer-turn latency (ms):')
print(f'  Without cache - Mean: {avg_latency_no:.1f}')
print(f'  With cache - Mean: {avg_latency_with:.1f}')
print(f'\nLatency reduction: {(1 - avg_latency_with / avg_latency_no) * 100:.1f}%')
print(f'\nScaling effect:')
print(f'  Turn 1: No cache benefit')
print(f'  Turn 5: ~80% latency reduction (reuse 4 turns of history)')

## Comparison: Cache Hit Rate Strategies

In [ ]:
# Compare different prefix strategies
strategies_cache = {
    'Full sequence': {'prefix_len': 'all', 'hit_rate': 0.15, 'speedup': 1.2},
    'System prompt only': {'prefix_len': 100, 'hit_rate': 0.40, 'speedup': 1.5},
    'System + first doc': {'prefix_len': 400, 'hit_rate': 0.25, 'speedup': 1.8},
    'Sliding window': {'prefix_len': 'dynamic', 'hit_rate': 0.55, 'speedup': 2.1},
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

names = list(strategies_cache.keys())
hit_rates = [strategies_cache[n]['hit_rate'] for n in names]
speedups = [strategies_cache[n]['speedup'] for n in names]
colors_cache = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

axes[0].bar(names, hit_rates, color=colors_cache, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Cache Hit Rate')
axes[0].set_title('Cache Hit Rates by Strategy')
axes[0].set_ylim([0, 0.7])
axes[0].grid(alpha=0.3, axis='y')
for i, v in enumerate(hit_rates):
    axes[0].text(i, v + 0.02, f'{v:.0%}', ha='center', fontweight='bold')

axes[1].bar(names, speedups, color=colors_cache, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Latency Speedup')
axes[1].set_title('Speedup vs No Caching')
axes[1].grid(alpha=0.3, axis='y')
for i, v in enumerate(speedups):
    axes[1].text(i, v + 0.05, f'{v:.1f}x', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## Key Takeaways

In [ ]:
print('='*70)
print('CACHE-AWARE SCHEDULING BEST PRACTICES')
print('='*70)
print('')
print('1. PREFIX HASHING')
print('   - Use SHA-256(first_k_tokens) for prefix identification')
print('   - k = system_prompt_len + avg_context_len')
print('   - Store hash → block_id mapping in LRU cache')
print('')
print('2. KV CACHE BLOCKS')
print('   - Block size: 16 tokens (balance fragmentation vs granularity)')
print('   - Reference counting for multi-request sharing')
print('   - Copy-on-write when sequence diverges')
print('')
print('3. ACHIEVABLE HIT RATES')
print('   - RAG (many docs, few unique): 40-60% hit rate')
print('   - Multi-turn (growing history): 50-70% hit rate')
print('   - General workload: 20-40% hit rate')
print('')
print('4. EXPECTED SPEEDUPS')
print('   - Hit: 40-50% latency reduction')
print('   - Typical improvement: 1.5-2.5x for high-overlap workloads')
print('')
print('5. MEMORY OVERHEAD')
print('   - Hash table: ~O(unique_prefixes * 20 bytes)')
print('   - Typical: <100MB even for 10K unique prefixes')
print('='*70)

## Exercises

In [ ]:
print('Exercise 1: Implement LRU Eviction')
print('-' * 70)
print('When KV cache memory is full, which prefix blocks should be evicted?')
print('  Naive: LRU (least recently used)')
print('  Better: LRU weighted by reference count (keep popular prefixes)')
print('  Best: Predict future hit probability')
print('')
print('Exercise 2: Prefix Length Tuning')
print('-' * 70)
print('Short prefix: More hits, but lose semantic context')
print('Long prefix: Fewer hits, but better quality')
print('How do you find the optimal prefix length?')
print('Hint: Plot hit_rate vs latency_improvement, find knee')
print('')
print('Success! Generated notebook 50 (cache-aware-scheduling)')